# Chapter 7 — Transforms

**Book alignment:** PyTorch From First Principles, Chapter 7

**Question this notebook isolates:** Does the uint8→float32 scale convention (`/255` vs no scaling) change the model-facing range — and hence first-epoch loss — while shapes, dtypes, and finiteness all look healthy?


In [ ]:
import numpy as np
import torch

torch.manual_seed(0)
np.random.seed(0)


## 1. Same shape, same dtype, 255x different meaning

`ToDtype(float32)` with `scale=False` converts the dtype but keeps values at `0…255`; with scaling they land in `0…1`. Both are finite float32 of identical shape.


In [ ]:
raw = torch.randint(0, 256, (64, 1, 8, 8), dtype=torch.uint8)
unscaled = raw.to(torch.float32)          # scale=False: dtype converts, values stay
scaled = raw.to(torch.float32) / 255.0    # scale=True: reinterpret the range
print(f"unscaled: range=[{unscaled.min():.1f}, {unscaled.max():.1f}] {tuple(unscaled.shape)} {unscaled.dtype}")
print(f"scaled:   range=[{scaled.min():.3f}, {scaled.max():.3f}] {tuple(scaled.shape)} {scaled.dtype}")


In [ ]:
assert unscaled.shape == scaled.shape == (64, 1, 8, 8)
assert unscaled.dtype == scaled.dtype == torch.float32
assert torch.isfinite(unscaled).all() and torch.isfinite(scaled).all()
assert unscaled.max().item() > 200, "values still on the 0-255 scale"
assert scaled.max().item() <= 1.0 and scaled.min().item() >= 0.0
print("every Chapter-2 check passes on the broken pipeline")


## 2. Normalization must be verified on the model-facing tensor

Standardizing the *scaled* representation with its own mean/std must yield ~zero mean and unit std; standardizing with mismatched statistics must not.


In [ ]:
mu, sd = scaled.mean(), scaled.std()
normed = (scaled - mu) / sd
print(f"stats of scaled: mu={mu:.4f} sd={sd:.4f}")
print(f"normalized: mean={normed.mean():.2e} std={normed.std():.4f}")
wrong = (unscaled - mu) / sd   # right statistics, wrong representation
print(f"unscaled through same stats: range=[{wrong.min():.1f}, {wrong.max():.1f}]")


In [ ]:
assert abs(normed.mean().item()) < 1e-4
assert abs(normed.std().item() - 1.0) < 1e-4
assert wrong.min().item() < -1.0 and wrong.max().item() > 100.0
print("normalization is a claim about a specific representation, not a ritual")


## 3. Representation changes optimization: same SGD, different learning

Labels from a single pixel (linearly separable). Same logistic model, SGD `lr = 1.0`, 200 full-batch steps: the scaled pipeline must learn (acc ≥ 0.95, loss < 0.3) while the unscaled one diverges (loss > 10).


In [ ]:
torch.manual_seed(2)
imgs = torch.randint(0, 256, (200, 1, 8, 8), dtype=torch.uint8)
lbl = (imgs[:, 0, 0, 0].to(torch.float32) > 127.5).long()

def run(feats, steps=200, lr=1.0):
    w = torch.zeros(64, requires_grad=True)
    b = torch.zeros((), requires_grad=True)
    first, last, lomax = None, None, 0.0
    for _ in range(steps):
        logits = feats @ w + b
        loss = torch.nn.functional.binary_cross_entropy_with_logits(logits, lbl.float())
        if first is None:
            first = loss.item()
        last = loss.item()
        lomax = max(lomax, last)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
    with torch.no_grad():
        acc = (((feats @ w + b) > 0).long() == lbl).float().mean().item()
        wmax = w.abs().max().item()
    return first, last, acc, wmax, lomax

fs = (imgs.to(torch.float32) / 255.0).reshape(200, 64)
fu = imgs.to(torch.float32).reshape(200, 64)
f1s, fls, accs, wmax_s, lomax_s = run(fs)
f1u, flu, accu, wmax_u, lomax_u = run(fu)
print(f"scaled:   first={f1s:.4f} last={fls:.4f} acc={accs:.3f} max|w|={wmax_s:.2f}")
print(f"unscaled: first={f1u:.4f} last={flu:.4f} acc={accu:.3f} max|w|={wmax_u:.1f} peak_loss={lomax_u:.1f}")


In [ ]:
assert accs >= 0.95, accs
assert fls < 0.3, fls
assert wmax_s < 50.0, wmax_s
assert lomax_u > 100.0, lomax_u
assert wmax_u > 500.0, wmax_u
print("identical model/optimizer/seed: representation alone decides learning")


## What we earned

The model trains on whatever preprocessing produced, not on your notion of the raw sample. Trace one known sample through each boundary and verify range, calibration, and semantics there — headline metrics cannot do it for you (an adaptive optimizer can even hide the bug).

Chapter 8 carries the trusted `[B, C, H, W]` tensor across the model boundary: what shape reaches the next layer?
